# 02 · Preparación COCO Estados Unidos

Detector activo: **RT-DETRv2-S** con **OpenVINO Runtime nativo en Python**; checkpoint de referencia `merve/license-plates-rtdetrv2`, con licencia Apache-2.0. Este notebook prepara COCO para ajuste y construye un test interno ampliado y separado por grupos antes de calcular métricas.

In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="usa"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Formato COCO y protocolo de partición

La fuente principal es **European + USA License plates v1**, exportada por Roboflow en COCO Segmentation. Para detección se usan las cajas COCO y se consolidan sus tres etiquetas en una clase, `license_plate`.

La exportación observada contiene 13.143 imágenes de entrenamiento aumentadas, 386 de validación y 106 de test. El protocolo conserva validación + test como parte del test final, añade un 15% de grupos de origen del entrenamiento (un representante por grupo) y reserva otra validación agrupada del entrenamiento restante. Todas las variantes aumentadas de un grupo reservado quedan fuera de entrenamiento.

El test resultante es interno al conjunto Roboflow y se separa de los datos usados para ajustar SIRAP. No demuestra separación por vehículo o sesión salvo que completes la plantilla opcional de grupos con esos identificadores.

In [ ]:
from src.train import (
    RF_FORMAT,
    RF_PROJECT,
    RF_TITLE,
    RF_VERSION,
    RF_WORKSPACE,
    _annotation_file,
    download_roboflow_dataset,
)

DATA_DIR = None
USE_ROBOFLOW = True

print(
    f"Fuente Roboflow: {RF_TITLE} | "
    f"{RF_WORKSPACE}/{RF_PROJECT} | versión {RF_VERSION} | {RF_FORMAT}"
)

if USE_ROBOFLOW:
    # Configura ROBOFLOW_API_KEY en Secrets de Colab.
    DATA_DIR = str(download_roboflow_dataset("/content/datasets"))

if DATA_DIR:
    for split in ("train", "valid", "test"):
        try:
            print(split, _annotation_file(DATA_DIR, split))
        except FileNotFoundError as exc:
            print(exc)

## Crear el test interno ampliado y evitar fugas

Esta celda conserva una imagen por grupo del test y la validación originales, reserva además un 15% de los grupos del entrenamiento para ampliar el test y crea una validación nueva con grupos del entrenamiento restante. El entrenamiento conserva las variantes aumentadas solo de los grupos que sí le corresponden.

Antes de dividir, agrupa variantes por nombre fuente, aplica los identificadores opcionales de vehículo/sesión y une grupos que tengan archivos idénticos por SHA-256. Luego comprueba que ningún grupo ni imagen exacta atraviese train, valid y test. El resumen guarda cantidades, semilla, procedencia y distribución de etiquetas originales.

La plantilla source_group_manifest.csv permite asociar diferentes archivos fuente al mismo vehículo o sesión. Sin IDs proporcionados, la garantía es por archivo/familia de aumentos y duplicado exacto, no por vehículo. No uses el test para elegir checkpoints, hiperparámetros ni umbrales.

In [ ]:
import csv
import hashlib
import json
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

if not DATA_DIR:
    raise RuntimeError("Descarga o asigna primero el dataset COCO a DATA_DIR.")

SOURCE_ROOT = Path(DATA_DIR).resolve()
INDEPENDENT_ROOT = SOURCE_ROOT.parent / "european-usa-license-plates-independent-coco"
GROUP_MANIFEST = SOURCE_ROOT / "source_group_manifest.csv"
ADDITIONAL_TEST_GROUP_FRACTION = 0.15
VALIDATION_GROUP_FRACTION = 0.15
MINIMUM_TEST_IMAGES = 400
SPLIT_SEED = 42

if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"No existe el dataset: {SOURCE_ROOT}")

def source_key_from_filename(filename):
    basename = Path(filename).name
    return basename.split(".rf.", 1)[0] if ".rf." in basename else Path(basename).stem

def resolve_source_image(source_folder, filename):
    basename = Path(filename).name
    candidates = [
        source_folder / filename,
        source_folder / basename,
        SOURCE_ROOT / filename,
        SOURCE_ROOT / basename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    matches = list(SOURCE_ROOT.rglob(basename))
    if matches:
        return matches[0].resolve()
    raise FileNotFoundError(f"No se encontró la imagen anotada: {filename}")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def annotation_signature(record):
    return sorted(
        tuple(round(float(value), 4) for value in annotation["bbox"])
        for annotation in record["annotations"]
    )

source_payloads = {}
source_records = {}
category_maps = {}
all_source_keys = set()

for split_name in ("train", "valid", "test"):
    annotation_path = _annotation_file(SOURCE_ROOT, split_name)
    payload = json.loads(annotation_path.read_text(encoding="utf-8"))
    source_payloads[split_name] = payload
    category_maps[split_name] = {
        int(category["id"]): category.get("name", str(category["id"]))
        for category in payload.get("categories", [])
    }
    annotations_by_image = defaultdict(list)
    for annotation in payload.get("annotations", []):
        image_id = int(annotation["image_id"])
        if "bbox" not in annotation or len(annotation["bbox"]) != 4:
            raise ValueError(f"Anotación sin bbox COCO válida en {annotation_path}")
        annotations_by_image[image_id].append(annotation)

    image_ids = {int(image["id"]) for image in payload.get("images", [])}
    orphan_ids = set(annotations_by_image) - image_ids
    if orphan_ids:
        raise ValueError(f"Hay anotaciones sin imagen en {split_name}: {sorted(orphan_ids)[:5]}")

    records = []
    for image_record in payload.get("images", []):
        source_key = source_key_from_filename(image_record["file_name"])
        all_source_keys.add(source_key)
        record = {
            "source_split": split_name,
            "source_folder": annotation_path.parent,
            "image": image_record,
            "annotations": annotations_by_image.get(int(image_record["id"]), []),
            "source_key": source_key,
        }
        image_path = resolve_source_image(annotation_path.parent, image_record["file_name"])
        record["source_path"] = image_path
        record["sha256"] = sha256_file(image_path)
        records.append(record)
    source_records[split_name] = records

# Crear o ampliar la plantilla para que sobreviva a una nueva versión del export.
if not GROUP_MANIFEST.is_file():
    with GROUP_MANIFEST.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["source_key", "group_id"])
        writer.writeheader()
        writer.writerows({"source_key": key, "group_id": ""} for key in sorted(all_source_keys))
    print("Plantilla de grupos creada:", GROUP_MANIFEST)

with GROUP_MANIFEST.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    if not {"source_key", "group_id"}.issubset(reader.fieldnames or []):
        raise ValueError("La plantilla debe tener las columnas source_key y group_id.")
    manifest_rows = list(reader)

manifest_by_key = {}
for row in manifest_rows:
    key = (row.get("source_key") or "").strip()
    group_id = (row.get("group_id") or "").strip()
    if not key:
        continue
    if key in manifest_by_key and manifest_by_key[key] != group_id:
        raise ValueError(f"source_key repetida con grupos distintos: {key}")
    manifest_by_key[key] = group_id

missing_keys = sorted(all_source_keys - set(manifest_by_key))
if missing_keys:
    with GROUP_MANIFEST.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["source_key", "group_id"])
        writer.writerows({"source_key": key, "group_id": ""} for key in missing_keys)
    print(f"Se añadieron {len(missing_keys)} claves nuevas a la plantilla.")

for records in source_records.values():
    for record in records:
        manual_group = manifest_by_key.get(record["source_key"], "")
        record["base_group_id"] = (
            f"manual:{manual_group}" if manual_group else f"source:{record['source_key']}"
        )

# Un mismo archivo exacto nunca puede quedar en particiones diferentes.
parent = {
    record["base_group_id"]: record["base_group_id"]
    for records in source_records.values()
    for record in records
}

def find_group(group_id):
    while parent[group_id] != group_id:
        parent[group_id] = parent[parent[group_id]]
        group_id = parent[group_id]
    return group_id

def union_groups(left, right):
    root_left, root_right = find_group(left), find_group(right)
    if root_left == root_right:
        return
    first, second = sorted((root_left, root_right))
    parent[second] = first

records_by_hash = defaultdict(list)
for records in source_records.values():
    for record in records:
        records_by_hash[record["sha256"]].append(record)

for digest, copies in records_by_hash.items():
    signatures = {tuple(annotation_signature(record)) for record in copies}
    if len(signatures) > 1:
        names = [record["image"]["file_name"] for record in copies[:5]]
        raise ValueError(f"El mismo archivo tiene bbox distintas: {names}")
    group_ids = sorted({record["base_group_id"] for record in copies})
    for group_id in group_ids[1:]:
        union_groups(group_ids[0], group_id)

for records in source_records.values():
    for record in records:
        record["group_id"] = f"group:{find_group(record['base_group_id'])}"

rng = random.Random(SPLIT_SEED)

def one_record_per_group(records):
    grouped = defaultdict(list)
    for record in records:
        grouped[record["group_id"]].append(record)
    selected = []
    for group_id in sorted(grouped):
        choices = sorted(grouped[group_id], key=lambda item: item["image"]["file_name"])
        selected.append(rng.choice(choices))
    return selected

# El test y la validación Roboflow se reservan juntos para el test final.
original_heldout = source_records["valid"] + source_records["test"]
test_base_records = one_record_per_group(original_heldout)
original_heldout_groups = {record["group_id"] for record in original_heldout}

train_by_group = defaultdict(list)
for record in source_records["train"]:
    if record["group_id"] not in original_heldout_groups:
        train_by_group[record["group_id"]].append(record)

available_train_groups = sorted(train_by_group)
if len(available_train_groups) < 3:
    raise ValueError("Quedan menos de tres grupos para crear train, valid y test.")

rng.shuffle(available_train_groups)
desired_extra_test_groups = max(
    round(len(available_train_groups) * ADDITIONAL_TEST_GROUP_FRACTION),
    max(0, MINIMUM_TEST_IMAGES - len(test_base_records)),
)
extra_test_group_count = min(desired_extra_test_groups, len(available_train_groups) - 2)
if len(test_base_records) + extra_test_group_count < MINIMUM_TEST_IMAGES:
    raise ValueError(
        "No hay suficientes grupos independientes para crear un test de "
        f"{MINIMUM_TEST_IMAGES} imágenes."
    )

extra_test_group_ids = set(available_train_groups[:extra_test_group_count])
remaining_train_group_ids = available_train_groups[extra_test_group_count:]
extra_test_records = one_record_per_group([
    record
    for group_id in extra_test_group_ids
    for record in train_by_group[group_id]
])

validation_group_count = max(
    1, round(len(remaining_train_group_ids) * VALIDATION_GROUP_FRACTION)
)
validation_group_count = min(validation_group_count, len(remaining_train_group_ids) - 1)
validation_group_ids = set(remaining_train_group_ids[:validation_group_count])

validation_records = one_record_per_group([
    record
    for group_id in validation_group_ids
    for record in train_by_group[group_id]
])
training_records = [
    record
    for group_id in remaining_train_group_ids
    if group_id not in validation_group_ids
    for record in train_by_group[group_id]
]
test_records = test_base_records + extra_test_records

# Quitar duplicados exactos dentro de train para no darles peso adicional.
seen_train_hashes = set()
deduplicated_training = []
for record in sorted(training_records, key=lambda item: item["image"]["file_name"]):
    if record["sha256"] not in seen_train_hashes:
        seen_train_hashes.add(record["sha256"])
        deduplicated_training.append(record)
duplicate_train_images_removed = len(training_records) - len(deduplicated_training)
training_records = deduplicated_training

split_records = {
    "train": training_records,
    "valid": validation_records,
    "test": test_records,
}
group_sets = {
    name: {record["group_id"] for record in records}
    for name, records in split_records.items()
}
hash_sets = {
    name: {record["sha256"] for record in records}
    for name, records in split_records.items()
}
for left, right in (("train", "valid"), ("train", "test"), ("valid", "test")):
    group_overlap = group_sets[left] & group_sets[right]
    hash_overlap = hash_sets[left] & hash_sets[right]
    if group_overlap or hash_overlap:
        raise RuntimeError(
            f"Fuga entre {left} y {right}: "
            f"{len(group_overlap)} grupos y {len(hash_overlap)} hashes."
        )

if len(test_records) < MINIMUM_TEST_IMAGES:
    raise RuntimeError(f"El test final solo tiene {len(test_records)} imágenes.")

marker = INDEPENDENT_ROOT / ".sirap_independent_split.json"
if INDEPENDENT_ROOT.exists():
    if not marker.is_file():
        raise FileExistsError(
            f"{INDEPENDENT_ROOT} existe y no fue creado por este notebook; "
            "no se sobrescribirá."
        )
    shutil.rmtree(INDEPENDENT_ROOT)
INDEPENDENT_ROOT.mkdir(parents=True)

source_licenses = next(
    (payload.get("licenses", []) for payload in source_payloads.values()
     if payload.get("licenses")),
    [],
)
manifest_rows = []
split_summary = {}

for split_name, records in split_records.items():
    split_folder = INDEPENDENT_ROOT / split_name
    split_folder.mkdir(parents=True, exist_ok=True)
    output_images = []
    output_annotations = []
    source_category_counts = Counter()
    annotation_id = 0

    for image_id, record in enumerate(
        sorted(records, key=lambda item: (item["source_split"], item["image"]["file_name"])),
        start=1,
    ):
        original_name = Path(record["image"]["file_name"]).name
        output_name = f"{record['source_split']}__{record['sha256'][:12]}__{original_name}"
        destination = split_folder / output_name
        try:
            destination.symlink_to(record["source_path"])
        except (OSError, NotImplementedError):
            shutil.copy2(record["source_path"], destination)

        image_out = dict(record["image"])
        image_out.update({
            "id": image_id,
            "file_name": output_name,
            "sirap_source_key": record["source_key"],
            "sirap_group_id": record["group_id"],
            "sirap_sha256": record["sha256"],
            "sirap_original_split": record["source_split"],
        })
        output_images.append(image_out)

        category_map = category_maps[record["source_split"]]
        for source_annotation in record["annotations"]:
            bbox = [float(value) for value in source_annotation["bbox"]]
            x, y, width, height = bbox
            if x < 0 or y < 0 or width <= 0 or height <= 0:
                raise ValueError(f"bbox COCO inválida en {original_name}: {bbox}")
            annotation_id += 1
            output_annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 1,
                "bbox": bbox,
                "area": float(source_annotation.get("area", width * height)),
                "iscrowd": int(source_annotation.get("iscrowd", 0)),
            })
            source_category_counts[
                category_map.get(int(source_annotation["category_id"]), "unknown")
            ] += 1

        manifest_rows.append({
            "split": split_name,
            "file_name": output_name,
            "source_file_name": record["image"]["file_name"],
            "source_key": record["source_key"],
            "group_id": record["group_id"],
            "sha256": record["sha256"],
            "original_split": record["source_split"],
            "boxes": len(record["annotations"]),
        })

    empty_images = sum(
        1 for image in output_images
        if not any(annotation["image_id"] == image["id"] for annotation in output_annotations)
    )
    split_summary[split_name] = {
        "images": len(output_images),
        "groups": len({record["group_id"] for record in records}),
        "plates": len(output_annotations),
        "empty_images": empty_images,
        "source_category_annotations": dict(sorted(source_category_counts.items())),
    }
    coco_output = {
        "info": {
            "description": "SIRAP USA grouped internal test for RT-DETRv2-S",
            "source_dataset": "European + USA License plates, Roboflow v1",
            "grouping": "manual vehicle/session id when supplied; otherwise source filename and exact SHA-256",
        },
        "licenses": source_licenses,
        "images": output_images,
        "annotations": output_annotations,
        "categories": [{"id": 1, "name": "license_plate", "supercategory": "license_plate"}],
    }
    (split_folder / "_annotations.coco.json").write_text(
        json.dumps(coco_output, indent=2, ensure_ascii=False), encoding="utf-8"
    )

with (INDEPENDENT_ROOT / "split_manifest.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "split", "file_name", "source_file_name", "source_key",
            "group_id", "sha256", "original_split", "boxes",
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

train_group_overlap_with_original_holdout = {
    record["group_id"]
    for record in source_records["train"]
    if record["group_id"] in original_heldout_groups
}
SPLIT_SUMMARY = {
    "source_dataset": "European + USA License plates",
    "roboflow_version": 1,
    "seed": SPLIT_SEED,
    "minimum_test_images": MINIMUM_TEST_IMAGES,
    "additional_test_group_fraction": ADDITIONAL_TEST_GROUP_FRACTION,
    "validation_group_fraction": VALIDATION_GROUP_FRACTION,
    "test_design": (
        "one representative per group from original valid+test, plus one "
        "representative per randomly reserved train source group"
    ),
    "grouping_level": (
        "manual vehicle/session ids where supplied, otherwise source filename "
        "family; exact SHA-256 duplicates are merged"
    ),
    "manual_source_keys_grouped": sum(bool(value) for value in manifest_by_key.values()),
    "source_images": {name: len(records) for name, records in source_records.items()},
    "train_groups_removed_for_original_valid_test_overlap": len(train_group_overlap_with_original_holdout),
    "additional_train_groups_reserved_for_test": extra_test_group_count,
    "validation_groups_reserved_from_remaining_train": validation_group_count,
    "exact_duplicate_train_images_removed": duplicate_train_images_removed,
    "unique_sha256_audited": len({record["sha256"] for records in split_records.values() for record in records}),
    "splits": split_summary,
}
(INDEPENDENT_ROOT / "split_summary.json").write_text(
    json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False), encoding="utf-8"
)
marker.write_text(
    json.dumps({"generated_by": "SIRAP USA grouped COCO split", "seed": SPLIT_SEED}, indent=2),
    encoding="utf-8",
)

DATA_DIR = str(INDEPENDENT_ROOT)
INDEPENDENT_TEST_READY = True
print(json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False))
print("Nuevo DATA_DIR para entrenamiento y evaluación:", DATA_DIR)
print("Plantilla opcional de vehículo/sesión:", GROUP_MANIFEST)

## Evaluación final en OpenVINO

La evaluación se deja apagada deliberadamente. Primero entrena con train, elige checkpoint y umbral usando valid, y congela la configuración. Después registra esa decisión y ejecuta una sola evaluación final sobre test. La salida incluye el protocolo y el tamaño del test; si el archivo final ya existe, el notebook bloquea una segunda lectura accidental del test.

Reporta siempre el número de imágenes, grupos y placas junto a mAP50-95, AP50, AP75, precisión y recall. Si completaste grupos de vehículo/sesión, calcula también intervalos de confianza por bootstrap agrupado; si solo hay nombres de archivo, aclara que la incertidumbre por vehículo no está estimada.

In [ ]:
from pathlib import Path
import json
from src.evaluate import evaluate_detector

RUN_FINAL_TEST_EVALUATION = False
MODEL_CHECKPOINT_FROZEN = False
MODEL_SELECTION_RECORD = ""  # Ej.: checkpoint elegido por mAP50-95 de valid, commit y fecha.
FINAL_OPENVINO_CHECKPOINT = REPO_ROOT / "models/usa/detector/rtdetrv2_s/openvino"
FINAL_TEST_METRICS_PATH = REPO_ROOT / "results/metrics/rtdetr_usa_independent_test.json"

if RUN_FINAL_TEST_EVALUATION:
    if not globals().get("INDEPENDENT_TEST_READY", False):
        raise RuntimeError("Primero crea y audita el split independiente.")
    if not MODEL_CHECKPOINT_FROZEN or not MODEL_SELECTION_RECORD.strip():
        raise RuntimeError("Registra y congela el modelo seleccionado usando solo valid.")
    if not FINAL_OPENVINO_CHECKPOINT.is_dir():
        raise FileNotFoundError(f"No existe el checkpoint OpenVINO: {FINAL_OPENVINO_CHECKPOINT}")
    if FINAL_TEST_METRICS_PATH.exists():
        raise FileExistsError(
            f"Ya existe {FINAL_TEST_METRICS_PATH}; el test final no se vuelve a consultar."
        )
    if SPLIT_SUMMARY["splits"]["test"]["images"] < MINIMUM_TEST_IMAGES:
        raise RuntimeError("El test es menor al mínimo configurado.")

    FINAL_TEST_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_metrics_path = FINAL_TEST_METRICS_PATH.with_suffix(".working.json")
    if temporary_metrics_path.exists():
        temporary_metrics_path.unlink()

    final_test_metrics = evaluate_detector(
        data_dir=DATA_DIR,
        country="usa",
        split="test",
        checkpoint=FINAL_OPENVINO_CHECKPOINT,
        model_id=MODEL_ID,
        threshold=0.001,
        result_path=temporary_metrics_path,
    )
    final_test_metrics["test_protocol"] = {
        "source_dataset": "European + USA License plates, Roboflow v1",
        "seed": SPLIT_SEED,
        "test_design": SPLIT_SUMMARY["test_design"],
        "test_images": SPLIT_SUMMARY["splits"]["test"]["images"],
        "test_groups": SPLIT_SUMMARY["splits"]["test"]["groups"],
        "test_plates": SPLIT_SUMMARY["splits"]["test"]["plates"],
        "grouping_level": SPLIT_SUMMARY["grouping_level"],
        "model_selection_record": MODEL_SELECTION_RECORD,
        "threshold_for_ap": 0.001,
    }
    FINAL_TEST_METRICS_PATH.write_text(
        json.dumps(final_test_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    temporary_metrics_path.unlink(missing_ok=True)
    print(json.dumps(final_test_metrics, indent=2, ensure_ascii=False))
else:
    print(
        "Test final aún no evaluado. Ajusta y selecciona el modelo con train/valid; "
        "congélalo y luego activa RUN_FINAL_TEST_EVALUATION una sola vez."
    )